In [ ]:
import sys
print(sys.version)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms
from PIL import Image
from torchvision.models import resnet18, ResNet18_Weights, vit_b_16, ViT_B_16_Weights,  resnet50, ResNet50_Weights
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
from collections import Counter
from tqdm import tqdm
import torchvision.transforms.functional as TF
import gcsfs
import io
import json
from diffusers import DiffusionPipeline
from google.cloud import storage
import gc
import shap

In [ ]:
# =============================================================================
# Cloud & Hardware Initialization
# Sets up the Google Cloud Storage filesystem and detects whether a GPU (CUDA) 
# is available for hardware acceleration, falling back to CPU if not.
# =============================================================================
fs = gcsfs.GCSFileSystem()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

# Verify access to the GCS bucket by listing its root contents
fs.ls('full-image-data')

In [ ]:
# =============================================================================
# Synthetic "Not Faces" Dataset Generation
# Uses Stable Diffusion models (3.5 and XL) via Hugging Face diffusers to 
# generate a diverse set of synthetic non-face images based on hardcoded prompts.
# Images and their generation metadata are uploaded directly to GCS.
# =============================================================================
os.environ["HF_TOKEN"] = "XXX"
bucket_name = "full-image-data"

# Generation parameters
img_height = 1024
img_width = 1024
steps = 30
guidance = 7.5

# Diverse prompts to ensure the model learns robust non-face features
prompts = [
    "A cinematic portrait of an elderly fisherman with weathered skin at golden hour",
    "A studio headshot of a woman with neon face paint under UV light",
    "A candid street photo of a teenager laughing in the rain",
    "A renaissance-style oil painting of a royal queen in ornate armor",
    "A cyberpunk hacker illuminated by multiple holographic screens",
    "A black and white photo of a jazz singer in a smoky club",
    "A hyperrealistic portrait of an astronaut inside a space station",
    "A medieval knight covered in mud after battle",
    "A fashion magazine portrait of a model wearing avant-garde clothing",
    "A documentary-style photo of a farmer in rural India",
    "A vast desert with giant alien monoliths at sunset",
    "A snowy mountain range during a blizzard",
    "A peaceful Japanese garden during cherry blossom season",
    "A tropical beach during a thunderstorm",
    "A Scandinavian forest covered in morning fog",
    "A volcanic eruption at night with glowing lava rivers",
    "A futuristic floating city above the clouds",
    "An abandoned amusement park overtaken by nature",
    "A canyon illuminated by bioluminescent plants",
    "A northern lights display over a frozen lake",
    "A brutalist concrete building under dramatic lighting",
    "A glass skyscraper reflecting a sunset skyline",
    "A gothic cathedral interior with stained glass",
    "A small cozy cabin in the woods in winter",
    "A futuristic underground metro station",
    "A Moroccan marketplace with colorful textiles",
    "A Roman amphitheater during a historical reenactment",
    "A cyberpunk alleyway with neon signs and rain puddles",
    "A treehouse village high in a jungle canopy",
    "A luxury penthouse apartment interior",
    "A dragon perched on a castle tower at dusk",
    "A wizard casting a glowing spell in a dark forest",
    "A giant sea serpent attacking a wooden ship",
    "A fairy village inside a hollow tree",
    "A knight fighting a shadow monster",
    "A floating island with waterfalls spilling into the sky",
    "A magical library with books flying in the air",
    "A phoenix rising from flames",
    "A sorceress standing in a glowing magic circle",
    "A crystal cave illuminated by glowing gems",
    "A Mars colony under a transparent dome",
    "A humanoid robot cooking in a modern kitchen",
    "A spaceship landing on an alien planet",
    "A cybernetic soldier in a dystopian city",
    "A futuristic hospital with AI doctors",
    "A space battle between massive starships",
    "A post-apocalyptic city covered in vines",
    "A time traveler stepping through a glowing portal",
    "A sentient AI hologram floating in a lab",
    "A floating marketplace in zero gravity",
    "A barista making latte art in a cozy café",
    "A child flying a kite in a grassy field",
    "A family cooking dinner together",
    "A dog running through autumn leaves",
    "A crowded subway during rush hour",
    "A cyclist riding through a European city",
    "A picnic in a sunny park",
    "A chef plating a gourmet dish",
    "A rainy day in a small town",
    "A bookstore with warm lighting",
    "A lion resting under an acacia tree",
    "A polar bear walking across sea ice",
    "A hummingbird mid-flight",
    "A wolf howling under a full moon",
    "A coral reef teeming with colorful fish",
    "A bald eagle soaring above mountains",
    "A herd of elephants crossing a river",
    "A close-up macro photo of a butterfly",
    "A tiger walking through tall grass",
    "A panda eating bamboo",
    "An impressionist painting of a summer meadow",
    "A watercolor painting of a coastal village",
    "A cubist portrait of a musician",
    "A minimalist flat illustration of a city skyline",
    "A claymation-style character in a bakery",
    "A low-poly 3D render of a forest",
    "A hand-drawn pencil sketch of a violin",
    "A pixel art cyberpunk city",
    "A surrealist painting of melting clocks in a forest",
    "A charcoal drawing of a dancer",
    "A portrait lit only by candlelight",
    "A city skyline at blue hour",
    "A silhouette of a runner at sunrise",
    "A dramatic spotlight on a stage performer",
    "A car illuminated by neon street signs",
    "A mountain lit by lightning during a storm",
    "A forest with god rays shining through trees",
    "A product photo with studio softbox lighting",
    "A night market illuminated by lanterns",
    "A kitchen lit by early morning sunlight",
    "A bustling medieval marketplace with hundreds of people",
    "A futuristic airport terminal interior",
    "A pirate ship caught in a massive storm",
    "A royal coronation ceremony inside a palace",
    "A scientific laboratory with researchers working",
    "A grand ballroom dance scene",
    "A fantasy battlefield with armies clashing",
    "A wildlife documentary scene of predators hunting",
    "A festival celebration with fireworks",
    "A panoramic view of a megacity at night"
]
num_images = len(prompts)

storage_client = storage.Client()
bucket = storage_client.bucket(bucket_name)

def upload_image_to_gcs(image: Image.Image, path: str):
    """Saves a PIL image to an in-memory buffer and uploads it to GCS."""
    buffer = io.BytesIO()
    image.save(buffer, format="PNG")
    buffer.seek(0)
    blob = bucket.blob(path)
    blob.upload_from_file(buffer, content_type="image/png")

def upload_metadata_to_gcs(metadata: dict, path: str):
    """Uploads generation parameters as a JSON file to GCS."""
    blob = bucket.blob(path)
    blob.upload_from_string(
        json.dumps(metadata, indent=2),
        content_type="application/json"
    )

def generate_images(model_name, model_path):
    """Generates images iteratively using a specified diffusion model and pushes to GCS."""
    print(f"Loading {model_name}...")
    
    # Initialize the diffusion pipeline
    pipe = DiffusionPipeline.from_pretrained(
        model_path,
        torch_dtype=torch.float16,
        token=os.environ["HF_TOKEN"],
        device_map="balanced"
    )
    pipe.enable_attention_slicing() # Optimize memory usage
    
    for i in range(num_images):
        prompt = prompts[i]
        seed = i
        
        # Tie generation to a fixed seed for reproducibility
        generator = torch.Generator(device=device).manual_seed(seed)
        image = pipe(
            prompt,
            height=img_height,
            width=img_width,
            num_inference_steps=steps,
            guidance_scale=guidance,
            generator=generator
        ).images[0]
    
        # Define GCS paths
        image_path = f"nonface/{model_name}/img_{i+1}.png"
        metadata_path = f"nonface/{model_name}/img_{i+1}.json"
        
        # Upload assets
        upload_image_to_gcs(image, image_path)
        metadata = {
            "model": model_name,
            "prompt": prompt,
            "seed": seed,
            "steps": steps,
            "guidance_scale": guidance,
            "height": img_height,
            "width": img_width
        }
        upload_metadata_to_gcs(metadata, metadata_path)
        print(f"{model_name} image {i+1}/{num_images} uploaded")
        
    # Clear memory explicitly to avoid OOM errors between model loads
    del pipe
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

# Generate the datasets using two different SD architectures
generate_images(model_name="Stable Diffusion 3.5", model_path="stabilityai/stable-diffusion-3.5-large")
generate_images(model_name="Stable Diffusion XL", model_path="stabilityai/stable-diffusion-xl-base-1.0")
print("All images uploaded to GCS.")

In [ ]:
# Fourier transform (frequency of image)
def fft_transform(img_tensor):
    """
    Applies a 2D Fast Fourier Transform (FFT) to extract frequency domain features.
    Particularly useful to detect high-frequency artifacts left by AI image generators.
    
    img_tensor: (C, H, W), values in [0,1]
    returns: (C, H, W) magnitude spectrum
    """
    fft = torch.fft.fft2(img_tensor)
    fft = torch.fft.fftshift(fft) # Shift zero-frequency component to center
    mag = torch.log(torch.abs(fft) + 1e-8) # Calculate magnitude and apply log scale
    return mag

In [ ]:
class DualDataset(torch.utils.data.Dataset):
    """
    Loads .pt files from a GCS bucket. Each .pt file contains 'rgb' and 'freq' tensors.
    Bucket layout: gs://<gcs_bucket>/<gcs_path>/<class_label>/<file>.pt

    Augmentations (all optional, controlled by constructor flags):
      RGB:  hflip, vflip, rotation, gaussian_blur, color_jitter, random_erasing
      Freq: gaussian_noise, random_masking
      Both: zero out RGB or frequency, pick random subset (for training), pick many subsets (for testing)
    """
    def __init__(
        self,
        gcs_bucket,
        gcs_path,
        original_size = (1024, 1024),
        # whether to crop, patch, resize image
        patch_size=None,
        num_patches=1,
        zero_rgb_or_freq_p=0.0,
        rgb_size=None,
        freq_size=None,
        # normalization parameters
        rgb_mean=[0.485, 0.456, 0.406],
        rgb_std=[0.229, 0.224, 0.225],
        freq_mean=[0.0, 0.0, 0.0],
        freq_std=[1.0, 1.0, 1.0],
        # RGB-only augmentations
        rgb_trans_p = 0.0,
        rgb_hflip_p=0.0,
        rgb_vflip_p=0.0,
        rgb_max_rotation=0.0,
        rgb_blur_p=0.0,
        rgb_blur_sigma=(0.1, 2.0),
        rgb_color_jitter_p=0.0,
        rgb_color_jitter_strength=(0.3, 0.3, 0.2, 0.05),
        rgb_erasing_p=0.0,
        # Freq-only augmentations
        freq_noise_std=0.0,
        freq_noise_p = 0.0,
        freq_mask_p=0.0,
        freq_mask_size_frac=0.1,

    ):
        self.fs = None 

        self.original_size = original_size
        self.patch_size = patch_size
        self.num_patches = num_patches
        self.zero_rgb_or_freq_p = zero_rgb_or_freq_p
        self.rgb_size = rgb_size
        self.freq_size = freq_size

        self.rgb_mean = rgb_mean
        self.rgb_std = rgb_std
        self.freq_mean = freq_mean
        self.freq_std = freq_std

        self.rgb_trans_p = rgb_trans_p
        self.rgb_hflip_p = rgb_hflip_p
        self.rgb_vflip_p = rgb_vflip_p
        self.rgb_max_rotation = rgb_max_rotation
        self.rgb_blur_p = rgb_blur_p
        self.rgb_blur_sigma = rgb_blur_sigma
        self.rgb_color_jitter_p = rgb_color_jitter_p
        self.rgb_color_jitter = transforms.ColorJitter(*rgb_color_jitter_strength)
        self.rgb_erasing_p = rgb_erasing_p
        self.rgb_erasing = transforms.RandomErasing(p=1.0, scale=(0.02, 0.1))

        self.freq_noise_std = freq_noise_std
        self.freq_noise_p = freq_noise_p
        self.freq_mask_p = freq_mask_p
        self.freq_mask_size_frac = freq_mask_size_frac


        setup_fs = gcsfs.GCSFileSystem()
        
        # Build sample list
        root = f"{gcs_bucket}/{gcs_path}"
        all_dirs = sorted([p for p in setup_fs.ls(root) if setup_fs.isdir(p)])
        
        class_dirs = [c for c in all_dirs if not os.path.basename(c).startswith('closed_set')]
        self.classes = [os.path.basename(c) for c in class_dirs]
        self.class_to_idx = {cls: idx for idx, cls in enumerate(self.classes)}
        self.samples = []
        for class_dir in class_dirs:
            label = os.path.basename(class_dir)
            idx = self.class_to_idx[label]
            for fpath in setup_fs.ls(class_dir):
                if fpath.endswith('.png'):
                    self.samples.append((fpath, idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        if self.fs is None:
            self.fs = gcsfs.GCSFileSystem()

        path, target = self.samples[index]
        with self.fs.open(path, "rb") as f:
            img = Image.open(f).convert("RGB")
        rgb = TF.to_tensor(img)

        rgb_size_transform = transforms.Compose([transforms.Resize(self.original_size)])
        rgb = rgb_size_transform(rgb)
        
        H = rgb.shape[1]
        W = rgb.shape[2]

        if self.patch_size is not None:

            patch_H, patch_W = self.patch_size
            if H < patch_H or W < patch_W:
                raise ValueError(f"Image {path} ({H}x{W}) is smaller than patch_size ({patch_H}x{patch_W})")

            if self.num_patches == 1:
                bottom = random.randint(0, H - patch_H)
                left = random.randint(0, W - patch_W)

                rgb = rgb[:, bottom:bottom + patch_H, left:left + patch_W]
                freq = fft_transform(rgb)

            else:
                rgb_patches = []
                freq_patches = []

                for _ in range(self.num_patches):
                    bottom = random.randint(0, H - patch_H)
                    left = random.randint(0, W - self.patch_size[1])

                    patch = rgb[:, bottom:bottom + patch_H, left:left + patch_W]
                    rgb_patches.append(patch)
                    freq_patches.append(fft_transform(patch))

                rgb = torch.stack(rgb_patches, dim=0) # (num_patches, C, H, W)
                freq = torch.stack(freq_patches, dim=0)

        else:
            freq = fft_transform(rgb)

        # randomly zero out either rgb or freq
        rgb_zero = False
        freq_zero = False
        if random.random() < self.zero_rgb_or_freq_p:
            if random.random() < 0.5:
                rgb = torch.zeros_like(rgb)
                rgb_zero = True
            else:
                freq = torch.zeros_like(freq)
                freq_zero = True

        # resize rgb and/or freq if specified
        if self.rgb_size is not None:
            is_3d = (rgb.dim() == 3)
            rgb_input = rgb.unsqueeze(0) if is_3d else rgb
            rgb = F.interpolate(rgb_input, size=self.rgb_size, mode='bilinear', align_corners=False)
            if is_3d: rgb = rgb.squeeze(0)

        if self.freq_size is not None:
            is_3d = (freq.dim() == 3)
            freq_input = freq.unsqueeze(0) if is_3d else freq
            freq = F.interpolate(freq_input, size=self.freq_size, mode='bilinear', align_corners=False)
            if is_3d: freq = freq.squeeze(0)



        # RGB-only transforms (DO NOT USE IF BATCHED)
        if not rgb_zero:
            if random.random() < self.rgb_trans_p:
                if random.random() < self.rgb_hflip_p:
                    rgb = TF.hflip(rgb)
                if random.random() < self.rgb_vflip_p:
                    rgb = TF.vflip(rgb)
                if self.rgb_max_rotation > 0:
                    angle = random.uniform(-self.rgb_max_rotation, self.rgb_max_rotation)
                    rgb = TF.rotate(rgb, angle)
                if random.random() < self.rgb_blur_p:
                    sigma = random.uniform(*self.rgb_blur_sigma)
                    kernel = int(sigma * 4) | 1
                    rgb = TF.gaussian_blur(rgb, kernel_size=kernel, sigma=sigma)
                if random.random() < self.rgb_color_jitter_p:
                    rgb = self.rgb_color_jitter(rgb)
                if random.random() < self.rgb_erasing_p:
                    rgb = self.rgb_erasing(rgb)

            # normalize
            rgb_means = torch.tensor(self.rgb_mean).view(1,3,1,1)
            rgb_stds  = torch.tensor(self.rgb_std).view(1,3,1,1)
            if rgb.dim() == 3:
                rgb = (rgb - rgb_means.squeeze(0)) / rgb_stds.squeeze(0)
            else:
                rgb = (rgb - rgb_means) / rgb_stds

        # Freq-only transforms (DO NOT USE IF BATCHED)
        if not freq_zero:
            if random.random() < self.freq_noise_p:
                freq = freq + torch.randn_like(freq) * self.freq_noise_std
            if random.random() < self.freq_mask_p:
                _, H, W = freq.shape
                mh = int(H * self.freq_mask_size_frac)
                mw = int(W * self.freq_mask_size_frac)
                top = random.randint(0, H - mh)
                left = random.randint(0, W - mw)
                freq[:, top:top+mh, left:left+mw] = 0.0

            # normalize
            freq_means = torch.tensor(self.freq_mean).view(1,3,1,1)
            freq_stds  = torch.tensor(self.freq_std).view(1,3,1,1)
            if freq.dim() == 3:
                freq = (freq - freq_means.squeeze(0)) / freq_stds.squeeze(0)
            else:
                freq = (freq - freq_means) / freq_stds

        return rgb, freq, target

In [ ]:
# ---------------------------------------------------------
# DATASET SPLITTING & DATALOADER SETUP
# ---------------------------------------------------------
# Load once with no augmentation to get the full sample list
full_closed = DualDataset(
    gcs_bucket="full-image-data",
    gcs_path="closed_set", 
    
)

# Extract labels for stratification
labels = [label for _, label in full_closed.samples]

# Stratified split
indices = list(range(len(full_closed)))
train_idx, temp_idx, _, temp_labels = train_test_split(
    indices, labels, test_size=0.30, stratify=labels, random_state=42
)
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.50, stratify=temp_labels, random_state=42
)

# Now load three augmented versions, but override the sample list
train_ds = DualDataset("full-image-data", "closed_set", 
                       patch_size = None, zero_rgb_or_freq_p = 0.3, 
                       rgb_size = (224,224), freq_size = (224,224),
                       rgb_hflip_p=0.5, rgb_vflip_p=0.2, rgb_max_rotation=15.0,
                       rgb_blur_p=0.3, rgb_color_jitter_p=0.4, rgb_erasing_p=0.2,
                       freq_noise_std=0.01, freq_noise_p=0.4, freq_mask_p=0.05,
                      rgb_trans_p=0.4)
val_ds = DualDataset("full-image-data", "closed_set", 
                    patch_size = None, num_patches = 1, 
                     rgb_size = (224,224), freq_size = (224,224))
test_ds = DualDataset("full-image-data", "closed_set", 
                      patch_size = None, num_patches = 1, 
                     rgb_size = (224,224), freq_size = (224,224))

# Reuse the split indices so each dataset sees only its own samples
train_ds.samples = [full_closed.samples[i] for i in train_idx]
val_ds.samples = [full_closed.samples[i] for i in val_idx]
test_ds.samples = [full_closed.samples[i] for i in test_idx]

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=30, pin_memory=True, persistent_workers = True)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=30, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=30, pin_memory=True)

# Print counts per class for each split
idx_to_class = {v: k for k, v in full_closed.class_to_idx.items()}

def print_counts(name, samples):
    counts = Counter(label for _, label in samples)
    print(f"\n{name} ({sum(counts.values())} total):")
    for idx in sorted(counts):
        print(f"  {idx_to_class[idx]}: {counts[idx]}")

print_counts("Train", train_ds.samples)
print_counts("Val",   val_ds.samples)
print_counts("Test",  test_ds.samples)

In [ ]:
def show_train_samples(dataset, n=8, cols=4):
    """
    Visualizes a grid of RGB images from the dataset.
    Applies denormalization using standard ImageNet mean/std to correctly display the original colors.
    """
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    axes = axes.flatten()
    
    for i in range(n):
        rgb, freq, target = dataset[i]
        
        # Denormalize
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
        std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
        img = (rgb * std + mean).clamp(0, 1)
        
        axes[i].imshow(img.permute(1, 2, 0).numpy())
        axes[i].set_title(idx_to_class[target])
        axes[i].axis('off')
    
    for j in range(i+1, len(axes)):
        axes[j].axis('off')
    
    plt.tight_layout()
    plt.show()

show_train_samples(train_ds, n=8)

In [ ]:
def show_train_samples_freq(dataset, n=8, cols=4):
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    axes = axes.flatten()
    
    for i in range(n):
        rgb, freq, target = dataset[i]
        
        # freq is (C, H, W) — take mean across channels, then normalize to [0,1] for display
        freq_display = freq.mean(dim=0)  # (H, W)
        freq_display = (freq_display - freq_display.min()) / (freq_display.max() - freq_display.min() + 1e-8)
        
        axes[i].imshow(freq_display.numpy(), cmap='inferno')
        axes[i].set_title(idx_to_class[target])
        axes[i].axis('off')
    
    for j in range(i+1, len(axes)):
        axes[j].axis('off')
    
    plt.tight_layout()
    plt.show()

show_train_samples_freq(train_ds, n=8)

In [ ]:
def show_test_patches(dataset, sample_idx=0):
    rgb, freq, target = dataset[sample_idx]
    # rgb shape: (num_patches, C, H, W)
    
    num_patches = rgb.shape[0]
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
    
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    fig.suptitle(f"All patches for test sample 0 | class: {idx_to_class[target]}", fontsize=14)
    axes = axes.flatten()
    
    for i in range(min(num_patches,10)):
        patch = rgb[i]  # (C, H, W)
        img = (patch * std + mean).clamp(0, 1)
        axes[i].imshow(img.permute(1, 2, 0).numpy())
        axes[i].set_title(f"Patch {i+1}")
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()

show_test_patches(test_ds, sample_idx=0)

In [ ]:
## Baseline ViT (Same methodology as WILD paper)
# Vision Transformer model that adapts the classification head to our specific number of classes.
# Averages patch logits to produce a final aggregated score.
class ViTWILD(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.model = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1)
        
        head_input = self.model.heads[0].in_features
        self.model.heads = nn.Sequential(
            nn.Dropout(p=0.5, inplace=True),
            nn.Linear(head_input, num_classes)
        )

    def forward(self, rgb, freq=None):
        x = TF.resize(rgb, [224, 224], antialias=True)
        
        x = self.model._process_input(x)
        n = x.shape[0]

        batch_class_token = self.model.class_token.expand(n, -1, -1)
        x = torch.cat([batch_class_token, x], dim=1)
        x = x + self.model.encoder.pos_embedding

        x = self.model.encoder(x) 

        patch_embeddings = x[:, 1:]
        
        patch_logits = self.model.heads(patch_embeddings)
        
        final_aggregated_score = patch_logits.mean(dim=1)
        
        return final_aggregated_score

In [ ]:
## Baseline ResNet50 (Full Image)
# Standard ResNet50 architecture evaluated on the entire global image without patch extraction.
class ResNet50(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.model = resnet50(weights=ResNet50_Weights.DEFAULT)
        
        head_input = self.model.fc.in_features

        self.model.fc = nn.Sequential(
            nn.Dropout(p=0.5, inplace=True),
            nn.Linear(head_input, num_classes)
        )
        
    def forward(self, rgb, freq):
        return self.model(rgb)

In [ ]:
## ResNet50 WILD Implementation
# Evaluates images in patches during inference and averages the predictions to improve robustness.
class ResNet50WILD(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.model = resnet50(weights=ResNet50_Weights.DEFAULT)
        
        head_input = self.model.fc.in_features
        
        self.model.fc = nn.Sequential(
            nn.Dropout(p=0.5, inplace=True),
            nn.Linear(head_input, num_classes)
        )

    def forward(self, rgb, freq):
        if self.training:
            return self.model(rgb)

        batch_size, num_patches, C, H, W = rgb.shape
        

        rgb_reshaped = rgb.view(-1, C, H, W)
        
        patch_logits = self.model(rgb_reshaped)

        patch_outputs = patch_logits.view(batch_size, num_patches, -1)
        
        return patch_outputs.mean(dim=1)

In [ ]:
# Dual ResNet18 (Not used in the final paper)
# Experimental Dual-Stream model using lighter ResNet18 backbones for both streams to save compute.
class DualResNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        # default weights for the RGB model
        self.rgb_net = resnet18(weights=ResNet18_Weights.DEFAULT)
        
        # we will learn the weights for the frequency model
        self.freq_net = resnet18(weights=None)

        rgb_features = self.rgb_net.fc.in_features
        freq_features = self.freq_net.fc.in_features

        self.rgb_net.fc = nn.Identity()
        self.freq_net.fc = nn.Identity()

        # add a linear layer (ReLU activation) and dropout (to reduce overfitting)
        # can change process here
        self.classifier = nn.Sequential(
            nn.Linear(rgb_features + freq_features, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    # forward pass for our model
    def forward(self, rgb, freq):
        f_rgb = self.rgb_net(rgb)
        f_freq = self.freq_net(freq)
        return self.classifier(torch.cat([f_rgb, f_freq], dim=1))

In [ ]:
## Final Dual Fusion Model (Frozen RGB Backbone)
# Fuses a pre-trained, frozen Vision Transformer (RGB) with a trainable ResNet18 (Frequency).
# Designed to test if locking spatial features forces the model to learn frequency artifacts.
class DualStreamFusionFrozen(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        # default weights for the RGB model
        self.rgb_net = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1)
        
        for param in self.rgb_net.parameters():
            param.requires_grad = False

        # we will learn the weights for the frequency model
        self.freq_net = resnet18(weights=None)

        rgb_features = self.rgb_net.heads[0].in_features
        freq_features = self.freq_net.fc.in_features

        self.rgb_net.heads = nn.Identity()
        self.freq_net.fc = nn.Identity()

        # add a linear layer (ReLU activation) and dropout (to reduce overfitting)
        # can change process here
        self.classifier = nn.Sequential(
            nn.Linear(rgb_features + freq_features, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    # forward pass for our model
    def forward(self, rgb, freq):
        f_rgb = self.rgb_net(rgb)
        f_freq = self.freq_net(freq)
        return self.classifier(torch.cat([f_rgb, f_freq], dim=1))

In [ ]:
## Dual ResNet18 WILD Implementation (Not used in the final paper)
# Dual-Stream ResNet18 that processes patch predictions during inference and averages them.
class DualResNet18(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        
        self.rgb_net = resnet18(weights=ResNet18_Weights.DEFAULT)
        
        # we will learn the weights for the frequency model
        self.freq_net = resnet18(weights=None)

        rgb_features = self.rgb_net.fc.in_features
        freq_features = self.freq_net.fc.in_features

        self.rgb_net.fc = nn.Identity()
        self.freq_net.fc = nn.Identity()

        # add a linear layer (ReLU activation) and dropout (to reduce overfitting)
        # can change process here
        self.classifier = nn.Sequential(
            nn.Linear(rgb_features + freq_features, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, rgb, freq):
        if self.training:
            f_rgb = self.rgb_net(rgb)
            f_freq = self.freq_net(freq)
            return self.classifier(torch.cat([f_rgb, f_freq], dim=1))

        batch_size, num_patches, C, H, W = rgb.shape
        

        rgb_reshaped = rgb.view(-1, C, H, W)
        freq_reshaped = freq.view(-1, C, H, W)
        
        f_rgb_p = self.rgb_net(rgb_reshaped)
        f_freq_p = self.freq_net(freq_reshaped)

        patch_logits = self.classifier(torch.cat([f_rgb_p, f_freq_p], dim=1))

        all_patch_outputs = patch_logits.view(batch_size, num_patches, -1)
        
        return all_patch_outputs.mean(dim=1)

In [ ]:
##final dual resenet model
class DualResNetV2(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        
        self.rgb_net = resnet50(weights=ResNet50_Weights.DEFAULT)
        self.rgb_net.fc = nn.Identity() 
        
        self.freq_net = resnet50(weights=None)
        self.freq_net.fc = nn.Identity() 
        
        self.classifier = nn.Sequential(
            nn.Linear(2048 + 2048, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, rgb, freq):
        if self.training:
            f_rgb = self.rgb_net(rgb)
            f_freq = self.freq_net(freq)
            return self.classifier(torch.cat([f_rgb, f_freq], dim=1))

        batch_size, num_patches, C, H, W = rgb.shape
        

        rgb_reshaped = rgb.view(-1, C, H, W)
        freq_reshaped = freq.view(-1, C, H, W)
        
        f_rgb_p = self.rgb_net(rgb_reshaped)
        f_freq_p = self.freq_net(freq_reshaped)

        patch_logits = self.classifier(torch.cat([f_rgb_p, f_freq_p], dim=1))

        all_patch_outputs = patch_logits.view(batch_size, num_patches, -1)
        
        return all_patch_outputs.mean(dim=1)

In [ ]:
# ---------------------------------------------------------
# EVALUATION FUNCTION
# Calculates average loss, accuracy, and error rate over a given dataloader.
# ---------------------------------------------------------
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for rgb, freq, labels in loader:
            rgb = rgb.to(device)
            freq = freq.to(device)
            labels = labels.to(device)

            outputs = model(rgb, freq)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * labels.size(0)
            preds = outputs.argmax(dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / total
    accuracy = correct / total
    error = 1.0 - accuracy

    return avg_loss, accuracy, error

In [ ]:
# define a class that will allow us to stop training early if performance doesn't improve
class EarlyStopping:
    def __init__(self, patience=5, min_delta=1e-3):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float("inf")
        self.counter = 0
        self.should_stop = False

    def step(self, val_loss):
        # improvement
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            return True
        # no improvement
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
            return False

In [ ]:


mods = {
    "resnet50": ResNet50(num_classes=10).to(device),
    "dual_resnet": DualResNet(num_classes=10).to(device),
    "dual_fusion": DualStreamFusionFrozen(num_classes=10).to(device),
    "resnet50WILD": ResNet50WILD(num_classes=10).to(device), 
    "vitWILD": ViTWILD(num_classes=10).to(device),
    "dual_resnet_f": DualResNetV2(num_classes=10).to(device),
    "dual_resnet_18": DualResNet18(num_classes=10).to(device),


}



In [ ]:
def train_model(model, model_name, max_epochs=50, loader=train_loader, learning_rate = 1e-4):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=learning_rate)
    early_stopping = EarlyStopping(patience=5, min_delta=1e-4)
    best_model_path = f"best_{model_name}.pt"

    for epoch in range(max_epochs):
        model.train()
        train_loss = 0.0
        
        pbar = tqdm(loader, desc=f"[{model_name}] Epoch {epoch+1}/{max_epochs}")

        for rgb, freq, labels_batch in pbar: 
            rgb = rgb.to(device)
            freq = freq.to(device)
            labels_batch = labels_batch.to(device)

            optimizer.zero_grad(set_to_none=True)
            outputs = model(rgb, freq)
            loss = criterion(outputs, labels_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            train_loss += loss.item() * labels_batch.size(0)
            pbar.set_postfix(train_loss=f"{loss.item():.4f}")

        train_loss /= len(loader.dataset)

        val_loss, val_acc, val_err = evaluate(
            model, val_loader, criterion, device
        )

        improved = early_stopping.step(val_loss)
        if improved:
            torch.save(model.state_dict(), best_model_path)

        print(
            f"[{model_name}] Epoch {epoch+1:02d} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_acc:.4f}"
        )

        if early_stopping.should_stop:
            print(f"[{model_name}] Early stopping triggered.")
            break

In [ ]:
for name, model in mods.items():
    print(f"\n===== Training {name.upper()} =====")
    train_model(model, name)

In [ ]:

# Comprehensive evaluation script computing Balanced Accuracy and ROC-AUC across multiple models.
def evaluate_models(models_dict, test_loader, device):
    """
    Evaluates models using pre-patched data from the test_loader.
    Expects dataloader to yield (Batch, NumPatches, C, H, W) tensors.
    """
    results = []

    for name, model in models_dict.items():
        print(f"\nEvaluating {name}...")

        model.eval()

        all_preds = []
        all_labels = []
        all_probs = []

        with torch.no_grad():
            for rgb, freq, labels in tqdm(test_loader, desc=f"Testing {name}", leave=False):
                rgb = rgb.to(device)
                freq = freq.to(device)

               
                outputs = model(rgb, freq)

                probs = F.softmax(outputs, dim=1)
                _, preds = torch.max(outputs, 1)

                all_probs.append(probs.cpu().numpy())
                all_preds.append(preds.cpu().numpy())
                all_labels.append(labels.numpy())

        y_true = np.concatenate(all_labels)
        y_pred = np.concatenate(all_preds)
        y_probs = np.concatenate(all_probs)

        bal_acc = balanced_accuracy_score(y_true, y_pred)

        try:
            roc_auc = roc_auc_score(y_true, y_probs, multi_class='ovr', average='macro')
        except ValueError:
            print(f"Warning: ROC-AUC failed for {name} (likely missing classes in test set).")
            roc_auc = float('nan')

        results.append({
            "Model": name,
            "Balanced Acc": bal_acc,
            "ROC-AUC": roc_auc
        })

    df = pd.DataFrame(results)

    df_display = df.copy()
    df_display["Balanced Acc"] = df_display["Balanced Acc"].map("{:.2%}".format)
    df_display["ROC-AUC"] = df_display["ROC-AUC"].map("{:.4f}".format)

    return df_display

In [ ]:
## Creating Post-Processed Data Loaders
# These loaders represent different test sets (Step 1, Step 2, Step 3, Non-Face) to evaluate
# the models against varying degrees of data distribution shifts and corruptions.
patches = None
n_patches = 1
size = (224,224)

def create_eval_loader(gcs_path):
    ds = DualDataset(
        gcs_bucket="full-image-data", 
        gcs_path=gcs_path, 
        patch_size=patches, 
        num_patches=n_patches, 
        rgb_size=size, 
        freq_size=size
    )
    return DataLoader(ds, batch_size=32, shuffle=False, num_workers=30, pin_memory=True)

step1_loader = create_eval_loader("closed_set_step1")
step2_loader = create_eval_loader("closed_set_step2")
step3_loader = create_eval_loader("closed_set_step3")
nonface_loader = create_eval_loader("nonface")


In [ ]:
## Evaluating against closed set & corrupted variations
# Iterates through each evaluation step (Plain, Step 1, Step 2, Step 3, Not Faces)
# to build out the final results tables for Balanced Accuracy and ROC-AUC.

steps_config = [
    ("Plain", test_loader), 
    ("Step 1", step1_loader),
    ("Step 2", step2_loader),
    ("Step 3", step3_loader), 
    ("Not Faces", nonface_loader)
]


df_bal = pd.DataFrame()
df_roc = pd.DataFrame()

print(f"Starting Benchmark on {len(mods)} models...")

for step_name, step_loader in steps_config:
    print(f"Processing {step_name}...")
    

    step_results = evaluate_models(mods, step_loader, device)
    
    for index, row in step_results.iterrows():
        model = row['Model']
        
        # Balanced Accuracy Table
        df_bal.loc[model, step_name] = row['Balanced Acc']
        
        # ROC-AUC Table
        df_roc.loc[model, step_name] = row['ROC-AUC']

print("\n=== Balanced Accuracy ===")
print(df_bal)
print("\n=== ROC AUC ===")
print(df_roc)

In [ ]:
df_bal.to_csv('balanced_acc.csv', index=False)
df_roc.to_csv('roc_auc.csv', index=False)

In [ ]:
# ---------------------------------------------------------
# SHAP INTERPRETABILITY EXPERIMENT
# Uses SHAP (SHapley Additive exPlanations) to quantify how much the model relies on RGB vs. Frequency
# NOTE: DeepExplainer requires ReLU to have inplace=False, hence the helper function
# ---------------------------------------------------------
model = DualResNetV2(num_classes=10).to(device)
model.load_state_dict(torch.load("best_dual_resnet_v3.pt", map_location=device))
def remove_inplace_relu(module):
    for name, child in module.named_children():
        if isinstance(child, torch.nn.ReLU):
            setattr(module, name, torch.nn.ReLU(inplace=False))
        else:
            remove_inplace_relu(child)
remove_inplace_relu(model)
model.to(device)
model.eval()

class FusionClassifier(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.classifier = model.classifier

    def forward(self, x):
        return self.classifier(x)

def extract_features(model, loader, max_batches=None):
    
    features = []
    labels = []

    with torch.no_grad():
        for i, (rgb, freq, y) in enumerate(loader):

            rgb = rgb.to(device)
            freq = freq.to(device)

            # If validation/test loader uses patches
            if rgb.dim() == 5:
                B, P, C, H, W = rgb.shape
                rgb = rgb.view(-1, C, H, W)
                freq = freq.view(-1, C, H, W)

                f_rgb = model.rgb_net(rgb)
                f_freq = model.freq_net(freq)

                fused = torch.cat([f_rgb, f_freq], dim=1)
                fused = fused.view(B, P, -1).mean(dim=1)

            else:
                f_rgb = model.rgb_net(rgb)
                f_freq = model.freq_net(freq)
                fused = torch.cat([f_rgb, f_freq], dim=1)

            features.append(fused.cpu())
            labels.append(y)

            if max_batches and i >= max_batches:
                break

    return torch.cat(features), torch.cat(labels)

features, labels = extract_features(model, val_loader, max_batches=10)

print(features.shape)


fusion_model = FusionClassifier(model).to(device)

background = features[:100].to(device)

explainer = shap.DeepExplainer(fusion_model, background)

test_samples = features[100:200].to(device)

shap_values = explainer.shap_values(test_samples)

rgb_importance = shap_values[:, :2048, :].mean()
freq_importance = shap_values[:, 2048:, :].mean()

print("RGB contribution:", rgb_importance)
print("Frequency contribution:", freq_importance)

total = rgb_importance + freq_importance
print("RGB %:", rgb_importance / total)
print("Freq %:", freq_importance / total)

In [ ]:
# ---------------------------------------------------------
# MANUAL WEIGHT-BASED INTERPRETABILITY
# Computes an 'Effective Weight Matrix' by multiplying the classifier layers.
# This directly scores the contribution of the first 2048 dims (RGB) vs the last 2048 dims (Freq).
# ---------------------------------------------------------

model = DualResNetV2(num_classes=10).to(device)
model.load_state_dict(torch.load("best_dual_resnet_v3.pt", map_location=device))
model.eval()

features = []
labels = []

with torch.no_grad():
    for rgb, freq, y in val_loader:

        rgb = rgb.to(device)
        freq = freq.to(device)

        # forward through feature extractors
        rgb_feat = model.rgb_net(rgb)      # (B,2048)
        freq_feat = model.freq_net(freq)   # (B,2048)

        embedding = torch.cat([rgb_feat, freq_feat], dim=1)  # (B,4096)

        features.append(embedding.cpu())
        labels.append(y)

X = torch.cat(features).numpy()
y = torch.cat(labels).numpy()

print("Embedding shape:", X.shape)

state_dict = model.state_dict()

W1 = state_dict["classifier.0.weight"].cpu().numpy()  # (512,4096)
W2 = state_dict["classifier.3.weight"].cpu().numpy()  # (10,512)

W_eff = W2 @ W1   # (10,4096)

print("Effective weight matrix:", W_eff.shape)

rgb_feats = X[:, :2048]
freq_feats = X[:, 2048:]

rgb_contrib = []
freq_contrib = []

for i in range(len(X)):

    cls = y[i]

    w = W_eff[cls]

    rgb_score = np.abs(rgb_feats[i] * w[:2048]).sum()
    freq_score = np.abs(freq_feats[i] * w[2048:]).sum()

    rgb_contrib.append(rgb_score)
    freq_contrib.append(freq_score)

rgb_contrib = np.array(rgb_contrib)
freq_contrib = np.array(freq_contrib)

total = rgb_contrib + freq_contrib

rgb_ratio = rgb_contrib / total
freq_ratio = freq_contrib / total

print("RGB contribution:", rgb_ratio.mean())
print("Frequency contribution:", freq_ratio.mean())